# 官方成交量拆分 — 最小可行試抓與對帳（Pilot）

- **第 5 格（試抓）**：向證交所和櫃買中心的公開報表抓 8 個試點日，每個請求都保存原始回應和 SHA-256。
  - 抓取報表：每日收盤行情、盤後定價、零股、鉅額。
  - 端點都是「候選」，抓不到也算結果。
  - 不需要任何憑證，不呼叫富邦。
- **第 6 格（對帳）**：只讀既有的缺棒帳本，不讀寫任何資料集。
  - 先在「K 棒全到齊」的日子實測：官方總量扣掉哪些成分，才會剛好等於 5 分K加總。
  - 這個關係驗證通過後，才把有缺棒的日子分類。分類結果都只是候選，帳本狀態不變。
- 結果寫到 Colab 本機，再複製到 Drive 的 `_tw-market-data-lake/audits/official_volume_pilot_<時間>/`（新資料夾）。

In [ ]:
# 1. 參數
DATES = "2023-08-17,2024-01-02,2024-05-24,2024-12-16,2025-04-10,2025-06-23,2026-03-30,2026-06-23"
REPO_OWNER, REPO_NAME, BRANCH = "hh4832", "tw-market-data-lake", "main"
DATA_ROOT = "/content/drive/MyDrive/00Quant_Research/data"
LEDGER_DIR = f"{DATA_ROOT}/_tw-market-data-lake/audits/gap_ledger_20261026_163358_3e8eed0"
OUT_DIR = "/content/tw_lake_official_volume"
PROJECT_DIR = f"/content/{REPO_NAME}"

In [ ]:
# 2. 掛載 Google Drive
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# 3. clone / pull
import subprocess, os
if os.path.exists(f"{PROJECT_DIR}/.git"):
    subprocess.run(["git", "-C", PROJECT_DIR, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", PROJECT_DIR, "checkout", "-B", BRANCH, f"origin/{BRANCH}"], check=True)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, f"https://github.com/{REPO_OWNER}/{REPO_NAME}.git", PROJECT_DIR], check=True)
print("HEAD:", subprocess.check_output(["git", "-C", PROJECT_DIR, "rev-parse", "HEAD"], text=True).strip())

In [ ]:
# 4. 依賴
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml>=6", "tqdm>=4.66"], check=True)
def stream(args):
    env = dict(os.environ, PYTHONPATH=f"{PROJECT_DIR}/src", PYTHONUNBUFFERED="1")
    p = subprocess.Popen(args, cwd=PROJECT_DIR, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="")
    return p.wait()

In [ ]:
# 5. 官方報表試抓（約 8 日 x 11 個候選端點，含禮貌間隔約 5 分鐘）
import glob
print("exit", stream([sys.executable, "-m", "tw_lake.reference.official_volume", "--dates", DATES, "--out-dir", OUT_DIR]))
PROBE_DIR = sorted(glob.glob(f"{OUT_DIR}/official_volume_probe_*"))[-1]

In [ ]:
# 6. 對帳（唯讀）
print("exit", stream([sys.executable, "-m", "tw_lake.audit.volume_recon", "--probe-dir", PROBE_DIR,
                      "--ledger-dir", LEDGER_DIR, "--out-dir", OUT_DIR]))
RECON_DIR = sorted(glob.glob(f"{OUT_DIR}/volume_recon_*"))[-1]

In [ ]:
# 7. 複製到 Drive（新資料夾）並等待上傳
import shutil, os
dest = f"{DATA_ROOT}/_tw-market-data-lake/audits/official_volume_pilot_{os.path.basename(PROBE_DIR).split('probe_')[1]}"
shutil.copytree(PROBE_DIR, f"{dest}/{os.path.basename(PROBE_DIR)}")
shutil.copytree(RECON_DIR, f"{dest}/{os.path.basename(RECON_DIR)}")
print("copied to", dest)
drive.flush_and_unmount()
print("Drive flushed")